# Chọn embedding model, nén vector, và kế hoạch migrate index

Bài mở: so ít nhất ba cấu hình embedding cho tìm điều luật tiếng Việt (model khác nhau, cắt bớt chiều, nén int8 và nhị
phân có rescoring), đo chất lượng, tốc độ, bộ nhớ, rồi viết kế hoạch đổi embedding model cho một index đang phục vụ mà
không downtime, trong `starter/REPORT.md`.

- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`. Phần đo là của bạn: thêm cell `explore` tuỳ ý.
- Cần cache `bge-m3.npy` (bài Intermediate 01) và Ollama với `bge-m3`, `nomic-embed-text`.
- Chấm hàm: `uv run pytest tracks/ai-engineer/modules/12-embeddings-vector-db/exercises/advanced/02-embedding-model-migration -q`

In [ ]:
# explore: tìm gốc repo, đường dẫn dữ liệu
import os
import sys
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "4")
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure

ZALO = ensure("zalo-legal")
CACHE = ZALO / "cache"

In [ ]:
import numpy as np

## 1. Nén vector: int8 và nhị phân

- `quantize_int8(x) -> (codes, scale)`: lượng tử hoá **đối xứng theo từng chiều**: `scale[j] = max_i |x[i, j]| / 127`
  (chiều toàn 0 thì `scale[j] = 1`), `codes = round(x / scale)` kẹp trong `[-127, 127]`, kiểu `int8`; `scale` float32
  shape `(d,)`. `dequantize_int8(codes, scale)` trả float32.
- `binarize(x) -> np.ndarray`: bit `1` khi phần tử `> 0`, đóng gói 8 bit một byte theo `np.packbits` mặc định (bit đầu
  là bit cao). Shape `(n, ceil(d / 8))` `uint8`.
- `hamming_topk(q_bits, db_bits, k) -> list[int]`: `k` chỉ số có khoảng cách Hamming nhỏ nhất tới `q_bits` (một dòng),
  bằng nhau thì chỉ số nhỏ.
- `rescore(q, candidates, x, k) -> list[int]`: xếp lại `candidates` (chỉ số dòng của `x`) theo tích vô hướng với `q`
  bằng vector gốc, giảm dần, bằng nhau thì chỉ số nhỏ; trả `k` chỉ số đầu.

In [ ]:
def quantize_int8(x: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """int8 đối xứng theo từng chiều: scale = max|x| / 127 (chiều toàn 0: 1)."""
    # TODO
    raise NotImplementedError


def dequantize_int8(codes: np.ndarray, scale: np.ndarray) -> np.ndarray:
    """codes * scale, float32."""
    # TODO
    raise NotImplementedError


def binarize(x: np.ndarray) -> np.ndarray:
    """Bit 1 khi > 0, np.packbits theo hàng."""
    # TODO
    raise NotImplementedError


def hamming_topk(q_bits: np.ndarray, db_bits: np.ndarray, k: int) -> list[int]:
    """k chỉ số Hamming nhỏ nhất, bằng nhau thì chỉ số nhỏ."""
    # TODO
    raise NotImplementedError


def rescore(q: np.ndarray, candidates, x: np.ndarray, k: int) -> list[int]:
    """Xếp lại candidates theo q · x (vector gốc), giảm dần, bằng nhau thì chỉ số nhỏ."""
    # TODO
    raise NotImplementedError

## 2. Đo (phần của bạn)

Gợi ý khung, chi tiết và tiêu chí ở `README.md` của bài:

1. Tập đánh giá: 300 câu hỏi ghim (300 query-id đầu của `qrels/test.jsonl`). Thước đo: Recall@10 và MRR@10 theo nhãn,
   như module 13 định nghĩa.
2. `bge-m3` trên cả corpus (cache): bản gốc float32, cắt còn 512 và 256 chiều (`truncate` của Basic 03), int8,
   nhị phân không rescoring, nhị phân + rescoring top-100.
3. Model thứ hai qua Ollama (`nomic-embed-text`, có và không có tiền tố `search_query: ` / `search_document: `) trên
   một **pool** nhỏ hơn: mọi điều của các văn bản chứa đáp án của 300 câu cộng một mẫu điều ngẫu nhiên; chạy `bge-m3`
   trên đúng pool đó để so công bằng. Văn bản dài vượt context của model thì cắt và ghi rõ.
4. Throughput embed (văn bản/giây) của từng model trên cùng máy, byte/vector của từng cách lưu.
5. Điền `REPORT.md`: bảng, lựa chọn, và kế hoạch migrate.